<h1 style=\"text-align: center; font-size: 50px;\"> 📦 Register Model </h1>

This notebook packages the **audio-native agentic workflow** as an **MLflow pyfunc model**, logs it with artifacts
(index, config), and registers it to the MLflow Model Registry for serving.

- Retrieval: **CLAP** audio embeddings over timestamped windows (+ **MMR** reranker)
- Generation: **Qwen Omni** listens to the selected audio windows and answers (no transcripts required)
- Orchestration: **LangGraph** (relevance → memory → retrieve → rerank → answer → memoize)
- Vector store: **FAISS** (in-model artifact or built on first run)
- Memory: disk-backed key-value cache (per-corpus+question)


# Notebook Overview

- Start Execution
- Install and Import Libraries
- Configure Settings
- Verify Assets
- KV Memory
- MLflow Registration
- Load Model and Test Payload
- Message History

# Start Execution

In [1]:
# Standard library imports
import os  # Provides OS-related utilities
import sys  # Allows manipulation of Python runtime environment
import time  # Enables time-based operations
from pathlib import Path  # Object-oriented file system paths

# Extend sys.path to allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.utils import (  # Utility functions for logging, LLM I/O, and schema generation
    load_config,
    load_secrets,
    load_secrets_to_env,
    get_project_root,
    logger,
)

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

# Install and Import Libraries

In [3]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 14 ms, sys: 8.01 ms, total: 22 ms
Wall time: 1.8 s


In [4]:
from __future__ import annotations  # Enables postponed evaluation of annotations (PEP 563)

# ─────── Standard Library ───────
import json  # JSON serialization and deserialization
import warnings  # Issue warning messages
from collections import namedtuple  # Factory for creating tuple subclasses with named fields
from datetime import datetime  # Date and time utilities
from pathlib import Path  # Object-oriented filesystem paths
from typing import Any, Dict, List, Literal, Optional, TypedDict  # Type hinting support
import numpy as np  # Numerical operations and array handling
import pandas as pd  # DataFrames for structured data

# ─────── Third-Party Packages ───────
import mlflow  # Model tracking and serving framework
import mlflow.pyfunc  # MLflow Python function interface for custom models
from mlflow.models.signature import ModelSignature  # Model signature
from mlflow.types.schema import Schema, ColSpec  # Schema definition
from IPython.display import Markdown, display  # IPython utilities for notebook output formatting

# ─────── Local application-specific imports ───────
from src.simple_kv_memory import SimpleKVMemory  # In-memory key-value store for agent state
from src.mlflow import Logger  # Logger from universal mlflow structure


# Configure Settings

In [5]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [6]:
project_root = get_project_root()

CONFIG_PATH = "../configs/config.yaml"
SECRETS_PATH = "../configs/secrets.yaml"
DEMO_FOLDER = "../demo"
MEMORY_PATH: Path = Path("../data/memory")

MLFLOW_EXPERIMENT_NAME = "AIStudio-Agentic-Audio-RAG-Experiment"
MLFLOW_RUN_NAME = "AIStudio-Agentic-Audio-RAG-Run"
MLFLOW_MODEL_NAME = "AIStudio-Agentic-Audio-RAG-Model"

In [7]:
# Load secrets from secrets.yaml file (if it exists) into environment
if Path(SECRETS_PATH).exists():
    load_secrets_to_env(SECRETS_PATH)
else:
    print(f"No secrets file found at {SECRETS_PATH}; relying on preexisting environment")

# Retrieve secrets from environment
try:
    secrets = load_secrets()
except ValueError:
    secrets = {}

# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")
print("✅ Secrets loaded successfully")

No secrets file found at ../configs/secrets.yaml; relying on preexisting environment
✅ Configuration loaded successfully
✅ Secrets loaded successfully


In [8]:
logger.info('Notebook execution started.')

## Verify Assets

In [9]:
def log_asset_status(asset_path: str, asset_name: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured.")
    else:
        logger.info(f"{asset_name} is not properly configured. Please ensure the required asset is correctly configured in your AI Studio project according to the README file.")

def log_secrets_status(secrets: Dict[str, Any], success_message: str, failure_message: str) -> None:
    """
    Logs the status of secrets based on their existence.

    Parameters:
        secrets (Dict[str, Any]): Secrets retrieved to check if they exist.
        success_message (str): Message to log if secrets exists.
        failure_message (str): Message to log if secrets do not exist.
    """
    if secrets:
        logger.info(f"Project secrets are available. {success_message}")
    else:
        logger.info(f"There are no project secrets found. {failure_message}")

In [10]:
log_asset_status(
    asset_path=CONFIG_PATH,
    asset_name="Config",
)

log_secrets_status(
    secrets=secrets,
    success_message="",
    failure_message="Please check if the secrets were propely connfigured in your secrets yaml file or in Secrets Manager."
)

# KV Memory

In [11]:
memory: SimpleKVMemory = SimpleKVMemory(MEMORY_PATH)
memory.set('dummy key', 'dummy value')

# MLflow Registration

In [12]:
%%time

from packaging.version import parse as vparse

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
print(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {MLFLOW_EXPERIMENT_NAME}")

MEMORY_PATH: Path = Path("../data/memory")

# === Get model path from config ===
model_path = config.get("model_path")
if model_path and os.path.exists(model_path):
    logger.info(f"✅ Model file found at: {model_path}")
else:
    logger.info(f"⚠️ Warning: Model file not found at {model_path}. Please verify the path in config.yaml.")

logger.info(f'Starting the experiment: {MLFLOW_EXPERIMENT_NAME}')
logger.info(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")

input_schema = Schema([
    ColSpec("string", "question"),
    ColSpec("string", "file_id", required=False),
    ColSpec("string", "audio_path", required=False),
])

output_schema = Schema([
    ColSpec("string", "question"),
    ColSpec("string", "file_id"),
    ColSpec("string", "answer"),
    ColSpec("string", "evidence"),
    ColSpec("boolean", "from_memory"),
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)

with mlflow.start_run(run_name=f"register-{MLFLOW_RUN_NAME}") as run:
    logger.info(f"Run's Artifact URI: {run.info.artifact_uri}")
    Logger.log_model(
        signature=signature,
        artifact_path=MLFLOW_MODEL_NAME,
        config_path=CONFIG_PATH,
        secrets_dict=secrets if secrets else None,
        model_path=model_path,
        demo_folder=DEMO_FOLDER,
    )
    
    # Construct the URI for the logged model
    model_uri = f"runs:/{run.info.run_id}/{MLFLOW_MODEL_NAME}"

    # Register the model into MLflow Model Registry
    mlflow.register_model(
        model_uri=model_uri,
        name=MLFLOW_MODEL_NAME
    )

print("Logged model at:", model_uri)
print("Registered name:", MLFLOW_MODEL_NAME)
logger.info(f"✅ Model registered successfully with run ID: {run.info.run_id}")

2026/04/02 12:15:02 INFO mlflow.tracking.fluent: Experiment with name 'AIStudio-Agentic-Audio-RAG-Experiment' does not exist. Creating a new experiment.


Using MLflow tracking URI: /phoenix/mlflow
Experiment: AIStudio-Agentic-Audio-RAG-Experiment


Successfully registered model 'AIStudio-Agentic-Audio-RAG-Model'.
2026/04/02 12:15:44 WARNING mlflow.tracking._model_registry.fluent: Run with id fdb733ba3b4a47b28455fe274d406692 has no artifacts at artifact path 'AIStudio-Agentic-Audio-RAG-Model', registering model based on models:/m-072cee148b4548a895376e5e9fdf6216 instead


Logged model at: runs:/fdb733ba3b4a47b28455fe274d406692/AIStudio-Agentic-Audio-RAG-Model
Registered name: AIStudio-Agentic-Audio-RAG-Model


Created version '1' of model 'AIStudio-Agentic-Audio-RAG-Model'.


CPU times: user 814 ms, sys: 40.5 s, total: 41.3 s
Wall time: 42.6 s


In [13]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).